# Person 6 — Line & Area charts (notebook)

**Line:** mean `overall` rating by `age` (no year column in cleaned data — age acts as the trend axis).  
**Area:** number of players per `age`.

Uses the same chunked load as `app.py` so large `data/cleaned_data.csv` files stay usable.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import plotly.express as px

# Project root = parent of notebooks/
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_CLEAN = BASE / "data" / "cleaned_data.csv"
DATA_SAMPLE = BASE / "data" / "sample_cleaned_data.csv"

USECOLS = ["age", "overall", "club_name", "nationality_name"]
CHUNK_ROWS = 200_000
MAX_ROWS = int(os.environ.get("FIFA_DASH_MAX_ROWS", "500_000"))


def resolve_data_path():
    if DATA_CLEAN.exists():
        return DATA_CLEAN
    if DATA_SAMPLE.exists():
        return DATA_SAMPLE
    raise FileNotFoundError("Add data/cleaned_data.csv or data/sample_cleaned_data.csv under the project root.")


def load_data(path: Path) -> pd.DataFrame:
    parts = []
    n = 0
    for chunk in pd.read_csv(path, usecols=USECOLS, chunksize=CHUNK_ROWS, low_memory=False):
        parts.append(chunk)
        n += len(chunk)
        if n >= MAX_ROWS:
            break
    df = pd.concat(parts, ignore_index=True)
    if len(df) > MAX_ROWS:
        df = df.iloc[:MAX_ROWS].copy()
    df = df.dropna(subset=["age", "overall", "club_name", "nationality_name"])
    df["age"] = df["age"].astype(int)
    df["overall"] = df["overall"].astype(int)
    return df


path = resolve_data_path()
df = load_data(path)
print(f"Loaded {path.name}: {len(df):,} rows (cap {MAX_ROWS:,}). Age range {df['age'].min()}–{df['age'].max()}.")
df.head()

## Chart 1 — Line: mean overall by age

In [ ]:
trend = df.groupby("age", as_index=False)["overall"].mean().sort_values("age")
fig_line = px.line(
    trend,
    x="age",
    y="overall",
    markers=True,
    title="Mean overall by age (trend — no year column in data)",
    template="plotly_white",
)
fig_line.update_traces(line=dict(width=3))
fig_line.update_layout(xaxis_title="Age", yaxis_title="Mean overall", height=420, hovermode="x unified")
fig_line.show()

## Chart 2 — Area: players per age

In [ ]:
counts = df.groupby("age").size().reset_index(name="players").sort_values("age")
fig_area = px.area(
    counts,
    x="age",
    y="players",
    title="Players per age (distribution)",
    template="plotly_white",
)
fig_area.update_traces(line_color="#636efa", fillcolor="rgba(99,110,250,0.35)")
fig_area.update_layout(xaxis_title="Age", yaxis_title="Player count", height=420)
fig_area.show()